In [1]:
# Importing Modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import clone

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"], bits=1024)
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"], bits=1024)
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/train/RT.csv")
rt_test_data = pd.read_csv("data/test/RT.csv")

# Generate RT FP
rt_train_fp = MorganFP(rt_train_data["smiles"], bits=1024)
rt_train_fp["smiles"] = rt_train_fp.index
rt_train_fp = rt_train_fp.merge(rt_train_data, on="smiles")
rt_test_fp = MorganFP(rt_test_data["smiles"], bits=1024)
rt_test_fp["smiles"] = rt_test_fp.index
rt_test_fp = rt_test_fp.merge(rt_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"], bits=1024)
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"], bits=1024)
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/train/B3DB.csv")
b3db_test_data = pd.read_csv("data/test/B3DB.csv")

# Generate B3DB FP
b3db_train_fp = MorganFP(b3db_train_data["smiles"], bits=1024)
b3db_train_fp["smiles"] = b3db_train_fp.index
b3db_train_fp = b3db_train_fp.merge(b3db_train_data, on="smiles")
b3db_test_fp = MorganFP(b3db_test_data["smiles"], bits=1024)
b3db_test_fp["smiles"] = b3db_test_fp.index
b3db_test_fp = b3db_test_fp.merge(b3db_test_data, on="smiles")

######################## DATA-5 ##################################
# Loading freesolv data
freesolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freesolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

# Generate freesolv FP
freesolv_train_fp = MorganFP(freesolv_train_data["smiles"], bits=1024)
freesolv_train_fp["smiles"] = freesolv_train_fp.index
freesolv_train_fp = freesolv_train_fp.merge(freesolv_train_data, on="smiles")
freesolv_test_fp = MorganFP(freesolv_test_data["smiles"], bits=1024)
freesolv_test_fp["smiles"] = freesolv_test_fp.index
freesolv_test_fp = freesolv_test_fp.merge(freesolv_test_data, on="smiles")

In [3]:
datasets_baseline_error = {"ESOL":0.53,
            "Lipophilicity":0.55,
            "RT":70.40,
            "B3DB":0.56,
            "FreeSolv":1.02}

In [4]:
# Function to run ML training and testing
def RunML(model_template, train_data, test_data, dataName, modelName):

	error_list = []
	num_runs = 3

	for i in range(num_runs):

		# Data
		X_train = train_data.drop(["smiles", "target"], axis=1).to_numpy()
		y_train = train_data["target"].to_numpy()
		X_test = test_data.drop(["smiles", "target"], axis=1).to_numpy()
		y_test = test_data["target"].to_numpy()

		model = clone(model_template)
	   
		if hasattr(model, 'random_state'):
			model.random_state = i + 1

		# Model training
		model.fit(X_train, y_train)

		# Model testing
		y_pred = model.predict(X_test)

		error_run = np.array(y_test) - np.array(y_pred)

		high_error = [i for i in error_run if abs(i) > datasets_baseline_error[dataName]]

		error_list.append((len(high_error)/len(y_pred)*100))

	error_mean, error_std = np.mean(error_list), np.std(error_list)
	error_str = f"{error_mean:.3f} ({error_std:.3f})"

	# Return results
	return pd.DataFrame({ 
		"Data": [dataName], 
		"Model": [modelName],
		"Error": [error_str]
	})

In [5]:
# Data dict
datasets = {"ESOL":{"train":esol_train_fp, "test":esol_test_fp},
            "Lipophilicity":{"train":lipophilicity_train_fp, "test":lipophilicity_test_fp},
            "RT":{"train":rt_train_fp, "test":rt_test_fp},
            "B3DB":{"train":b3db_train_fp, "test":b3db_test_fp},
            "FreeSolv":{"train":freesolv_train_fp, "test":freesolv_test_fp}}

# Model dict
model_dict = {
    "LR": LinearRegression(), 
    "SVM": SVR(),
    "RF": RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=16)
}

In [6]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        params_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_ML_{dataName}.csv")
        params = eval(params_df[params_df["Model"]== modelName]["Model Params"].to_list()[0])
        model =  model.set_params(**params)
        temp_out.append(RunML(model, data["train"], data["test"], dataName, modelName))

# Final output
ML_out = pd.concat(temp_out, ignore_index=True)
ML_out.to_csv("results/Output_ML_Error_Analysis.csv")
ML_out

,Data,Model,Error
0,ESOL,LR,83.186 (0.000)
1,Lipophilicity,LR,54.762 (0.000)
2,RT,LR,80.714 (0.000)
3,B3DB,LR,50.943 (0.000)
4,FreeSolv,LR,67.692 (0.000)
5,ESOL,SVM,53.982 (0.000)
6,Lipophilicity,SVM,36.905 (0.000)
7,RT,SVM,48.571 (0.000)
8,B3DB,SVM,18.868 (0.000)
9,FreeSolv,SVM,41.538 (0.000)
